# 01 · MoNuSeg: eksploracja danych i przygotowanie masek

Co robi ten notebook:
1. opisuje zbiór (liczności, narządy, rozmiary obrazów, liczba jąder),
2. pokazuje krok po kroku, jak jeden plik XML z poligonami staje się maską instancji do treningu,
3. robi to samo dla wszystkich obrazów (`preprocess()` = `uv run dvc repro`) i sprawdza błędy w adnotacjach,
4. opisuje rozmiary jąder (potrzebne do Cellpose i do cech FLS) oraz zamrożony podział train/val.

Kod: `src/histopathology_pnw/datasets/monuseg.py`. Dane: `uv run dvc pull`.

In [ ]:
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage.color import label2rgb
from skimage.measure import regionprops_table
from skimage.segmentation import find_boundaries

from histopathology_pnw.datasets import monuseg

pd.set_option("display.width", 160)
train = monuseg.list_split("train")
test = monuseg.list_split("test")
print(f"train: {len(train)} obrazów, test: {len(test)} obrazów")

## 1. Zbiór w liczbach

MoNuSeg 2018 (Kumar et al. 2017, 2020): kafle H&E 1000×1000 px z preparatów TCGA, powiększenie 40×,
jądra obrysowane ręcznie jako poligony. Używamy **oficjalnego podziału**: 37 obrazów treningowych
i 14 testowych. Zbiór testowy służy tylko do końcowej ewaluacji.

In [ ]:
shapes = {}
for s in train + test:
    image = monuseg.read_image(s.image_path)
    shapes[(s.split, image.shape, str(image.dtype))] = (
        shapes.get((s.split, image.shape, str(image.dtype)), 0) + 1
    )
pd.Series(shapes, name="liczba obrazów")

In [ ]:
organs = monuseg.organs()
ids = pd.DataFrame({"image_id": [s.image_id for s in train]})
ids = ids.merge(organs[["image_id", "organ"]], on="image_id", how="left").fillna(
    {"organ": "nieznany"}
)
print("Narządy w zbiorze treningowym (z PDF dołączonego do MoNuSeg; 13 obrazów nie ma w PDF):")
ids["organ"].value_counts()

Uwaga: PDF z MoNuSeg przy obrazach płuc (TCGA-18, -21, -38, -49, -50) podaje narząd „Liver”, a w kolumnie
choroby „Lung … carcinoma”. W `configs/monuseg_train_organs.csv` kolumna `organ` jest poprawiona na podstawie
choroby, a oryginał zostaje w `organ_in_pdf`.

## 2. Format adnotacji: XML z poligonami

Każde jądro to `Region` z listą wierzchołków `Vertex X=… Y=…` (współrzędne w pikselach, liczby
zmiennoprzecinkowe). Hierarchia: `Annotations/Annotation/Regions/Region/Vertices/Vertex`.

In [ ]:
sample = train[0]
root = ET.parse(sample.xml_path).getroot()
region = next(root.iter("Region"))
print("plik:", sample.xml_path.name)
print("element główny:", root.tag, "| pod nim:", [c.tag for c in root])
print("liczba regionów (jąder):", sum(1 for _ in root.iter("Region")))
print("atrybuty regionu:", {k: region.attrib[k] for k in ("Id", "Area", "Length")})
print("pierwsze wierzchołki:", [v.attrib for v in region.find("Vertices")[:3]])

## 3. Jeden obraz krok po kroku: XML → maska instancji

1. `read_polygons` czyta poligony jako tablice `(N, 2)` współrzędnych `(x, y)`.
2. `polygons_to_instance_mask` rasteryzuje każdy poligon `skimage.draw.polygon` (wiersz = y, kolumna = x),
   przycina go do obrazu i nadaje kolejną etykietę. Poligony z < 3 wierzchołkami są odrzucane.
   Gdy poligony się nakładają, **późniejszy w pliku wygrywa**; instancja całkiem przykryta znika.
3. Wynik: `uint16`, 0 = tło, 1..N = jądra. To format, którego oczekuje Cellpose (`train_labels`).

In [ ]:
image = monuseg.read_image(sample.image_path)
polygons = monuseg.read_polygons(sample.xml_path)
mask, stats = monuseg.polygons_to_instance_mask(polygons, image.shape[:2])
print(stats)

crop = np.s_[300:600, 300:600]  # wycinek 300x300 px, żeby było widać pojedyncze jądra
fig, ax = plt.subplots(1, 4, figsize=(20, 5.5))
ax[0].imshow(image)
ax[0].set_title(f"obraz {sample.image_id}")
ax[1].imshow(image[crop])
for p in polygons:
    ax[1].plot(p[:, 0] - 300, p[:, 1] - 300, lw=0.8, color="lime")
ax[1].set_xlim(0, 300)
ax[1].set_ylim(300, 0)
ax[1].set_title("wycinek + poligony z XML")
ax[2].imshow(label2rgb(mask[crop], bg_label=0))
ax[2].set_title("maska instancji (kolor = etykieta)")
overlay = image[crop].copy()
overlay[find_boundaries(mask[crop], mode="inner")] = (0, 255, 0)
ax[3].imshow(overlay)
ax[3].set_title("kontury z maski na obrazie")
for a in ax:
    a.axis("off")
plt.tight_layout()

## 4. Wszystkie obrazy

`preprocess()` robi to samo dla wszystkich 51 obrazów, zapisuje `data/processed/monuseg/<split>/<id>_masks.tif`
oraz `manifest.csv` ze statystykami. To jest ten sam kod, który uruchamia `uv run dvc repro`
(stage `preprocess_monuseg` w `dvc.yaml`).

In [ ]:
manifest = monuseg.preprocess()
manifest.groupby("split")[
    ["n_polygons", "n_too_few_vertices", "n_empty", "n_lost", "n_instances"]
].sum()

### Błędy w adnotacjach

Obrazy, w których coś zostało odrzucone albo poligony się nakładają. `n_overlap_pixels` to piksele
przypisane do więcej niż jednego jądra (rozstrzygane na korzyść późniejszego poligonu).

In [ ]:
issues = manifest[
    (manifest[["n_too_few_vertices", "n_empty", "n_lost"]].sum(axis=1) > 0)
    | (manifest["n_overlap_pixels"] > 0)
]
print(f"obrazy z jakimkolwiek problemem: {len(issues)} / {len(manifest)}")
issues.sort_values("n_overlap_pixels", ascending=False)

## 5. Liczba jąder na obraz

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for split, g in manifest.groupby("split"):
    ax.hist(g["n_instances"], bins=15, alpha=0.6, label=f"{split} (n={len(g)})")
ax.set_xlabel("liczba jąder na obraz")
ax.set_ylabel("liczba obrazów")
ax.legend()
manifest.groupby("split")["n_instances"].describe()

## 6. Rozmiary jąder

Pole i średnica równoważna (średnica koła o tym samym polu) z masek. Cellpose-SAM jest trenowany tak,
żeby działać bez podawania średnicy (`diameter=None`), ale rozkład warto znać: Cellpose 3 zakładał
~30 px, a te same wielkości (pole, okrągłość, solidność, ekscentryczność) będą cechami selektora FLS.

In [ ]:
props = []
for s in train + test:
    table = regionprops_table(
        monuseg.read_mask(s),
        properties=("area", "equivalent_diameter_area", "eccentricity", "solidity"),
    )
    props.append(pd.DataFrame(table).assign(split=s.split, image_id=s.image_id))
props = pd.concat(props, ignore_index=True)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(props["area"], bins=100, range=(0, 1500))
ax[0].set_xlabel("pole jądra [px²]")
ax[1].hist(props["equivalent_diameter_area"], bins=80, range=(0, 50))
ax[1].set_xlabel("średnica równoważna [px]")
props.groupby("split")[
    ["area", "equivalent_diameter_area", "eccentricity", "solidity"]
].describe().T

In [ ]:
tiny = props[props["area"] < 15]
print(
    f"jądra mniejsze niż 15 px (domyślne min_size w Cellpose): {len(tiny)} z {len(props)} ({len(tiny) / len(props):.2%})"
)

## 7. Podział train / val

Zbiór testowy jest nietykalny, więc walidacja pochodzi z 37 obrazów treningowych: losowy podział
**po obrazie** (seed 0) na 30 train i 7 val, zamrożony w `configs/monuseg_split.json` (w gicie).
W zadaniu I3 pulą do losowania jest 30 obrazów train, a val służy do ewaluacji w każdej rundzie.

In [ ]:
split = monuseg.load_train_val_split()
role = {i: "train" for i in split["train"]} | {i: "val" for i in split["val"]}
m = manifest[manifest["split"] == "train"].assign(role=lambda d: d["image_id"].map(role))
m = m.merge(ids, on="image_id")
print(m.groupby("role")["n_instances"].agg(["count", "sum", "mean"]))
m[m["role"] == "val"][["image_id", "organ", "n_instances"]]

## Podsumowanie

- 37 obrazów treningowych i 14 testowych, wszystkie 1000×1000×3 `uint8`. Train i test nie mają wspólnych obrazów.
- Liczba jąder: 24 140 poligonów w train, 6 697 w test; po rasteryzacji 24 130 i 6 697 instancji.
  Odrzucone (tylko train): 5 poligonów z < 3 wierzchołkami, 2 puste po przycięciu do obrazu, 3 jądra całkowicie
  przykryte przez późniejsze poligony.
- Nakładające się piksele są w **każdym** obrazie: sąsiednie poligony dzielą brzegi, a czasem zachodzą na siebie
  (do ok. 10 tys. px na obraz, czyli ~1% obrazu). Reguła „późniejszy wygrywa” przesuwa granicę o kilka pikseli,
  co ma znaczenie dla SQ/AJI, ale nie dla detekcji.
- Mediana średnicy jądra to ok. 20 px w train i 23 px w test. Jąder poniżej `min_size=15` px jest 0,02%.
- Maski instancji leżą w `data/processed/monuseg` (DVC). Podział train/val jest zamrożony w `configs/monuseg_split.json`.
- Następny krok: `02_oracle_inference.ipynb` (zadanie I1).